In [1]:
import pandas as pd  # main library for DataFrame operations
import gc            # garbage collector (allowed by subject)

In [2]:
# Read fines.csv from the ex05/data folder
df = pd.read_csv("data/fines.csv")

# Quick sanity check: first rows and column names
df.head(), df.columns

(      CarNumber  Refund   Fines    Make  Model  Year
 0  Y163O8161RUS     2.0  3200.0    Ford  Focus  1989
 1   E432XX77RUS     1.0  6500.0  Toyota  Camry  1995
 2   7184TT36RUS     1.0  2100.0    Ford  Focus  1984
 3  X582HE161RUS     2.0  2000.0    Ford  Focus  2015
 4  92918M178RUS     1.0  5700.0    Ford  Focus  2014,
 Index(['CarNumber', 'Refund', 'Fines', 'Make', 'Model', 'Year'], dtype='object'))

In [3]:
# Method 1: classic for-loop with range + iloc

def calc_with_for_loop(dataframe: pd.DataFrame):
    """
    Calculate fines / refund * year using a simple Python for-loop
    and .iloc to access each row.
    """
    result = []
    for i in range(len(dataframe)):
        row = dataframe.iloc[i]
        value = row["Fines"] / row["Refund"] * row["Year"]
        result.append(value)
    return result

In [4]:
%%timeit -n 3
# Use the for-loop implementation and store the result in a new column
df["calc_for_loop"] = calc_with_for_loop(df)

28.2 ms ± 3.7 ms per loop (mean ± std. dev. of 7 runs, 3 loops each)


In [7]:
df[["Fines", "Refund", "Year", "calc_for_loop"]].head()

,Fines,Refund,Year,calc_for_loop
0,3200.0,2.0,1989,3182400.0
1,6500.0,1.0,1995,12967500.0
2,2100.0,1.0,1984,4166400.0
3,2000.0,2.0,2015,2015000.0
4,5700.0,1.0,2014,11479800.0


In [8]:
# Method 2: calculation using DataFrame.iterrows()

def calc_with_iterrows(dataframe: pd.DataFrame):
    """
    Calculate fines / refund * year iterating over rows using iterrows().
    """
    result = []
    for _, row in dataframe.iterrows():
        value = row["Fines"] / row["Refund"] * row["Year"]
        result.append(value)
    return result

In [9]:
%%timeit -n 3
df["calc_iterrows"] = calc_with_iterrows(df)

25.3 ms ± 3.94 ms per loop (mean ± std. dev. of 7 runs, 3 loops each)


In [10]:
df[["Fines", "Refund", "Year", "calc_for_loop", "calc_iterrows"]].head()

,Fines,Refund,Year,calc_for_loop,calc_iterrows
0,3200.0,2.0,1989,3182400.0,3182400.0
1,6500.0,1.0,1995,12967500.0,12967500.0
2,2100.0,1.0,1984,4166400.0,4166400.0
3,2000.0,2.0,2015,2015000.0,2015000.0
4,5700.0,1.0,2014,11479800.0,11479800.0


In [11]:
# Method 3: use DataFrame.apply() with a row-wise function

def formula(row):
    """
    Row-wise formula: fines / refund * year.
    """
    return row["Fines"] / row["Refund"] * row["Year"]

In [12]:
%%timeit -n 3
df["calc_apply"] = df.apply(formula, axis=1)

7.78 ms ± 1.92 ms per loop (mean ± std. dev. of 7 runs, 3 loops each)


In [13]:
df[["Fines", "Refund", "Year", "calc_for_loop", "calc_iterrows", "calc_apply"]].head()

,Fines,Refund,Year,calc_for_loop,calc_iterrows,calc_apply
0,3200.0,2.0,1989,3182400.0,3182400.0,3182400.0
1,6500.0,1.0,1995,12967500.0,12967500.0,12967500.0
2,2100.0,1.0,1984,4166400.0,4166400.0,4166400.0
3,2000.0,2.0,2015,2015000.0,2015000.0,2015000.0
4,5700.0,1.0,2014,11479800.0,11479800.0,11479800.0


In [14]:
%%timeit -n 3
df["calc_series"] = df["Fines"] / df["Refund"] * df["Year"]

The slowest run took 9.44 times longer than the fastest. This could mean that an intermediate result is being cached.
678 µs ± 772 µs per loop (mean ± std. dev. of 7 runs, 3 loops each)


In [15]:
df[["Fines", "Refund", "Year", "calc_series"]].head()

,Fines,Refund,Year,calc_series
0,3200.0,2.0,1989,3182400.0
1,6500.0,1.0,1995,12967500.0
2,2100.0,1.0,1984,4166400.0
3,2000.0,2.0,2015,2015000.0
4,5700.0,1.0,2014,11479800.0


In [16]:
%%timeit -n 3
fines_values = df["Fines"].values
refund_values = df["Refund"].values
year_values = df["Year"].values

df["calc_values"] = fines_values / refund_values * year_values

The slowest run took 9.05 times longer than the fastest. This could mean that an intermediate result is being cached.
356 µs ± 316 µs per loop (mean ± std. dev. of 7 runs, 3 loops each)


In [17]:
df[["Fines", "Refund", "Year", "calc_values"]].head()

,Fines,Refund,Year,calc_values
0,3200.0,2.0,1989,3182400.0
1,6500.0,1.0,1995,12967500.0
2,2100.0,1.0,1984,4166400.0
3,2000.0,2.0,2015,2015000.0
4,5700.0,1.0,2014,11479800.0


In [18]:
cols = ["calc_for_loop", "calc_iterrows", "calc_apply", "calc_series", "calc_values"]

# Round a little to avoid tiny float differences
rounded = df[cols].round(6)

# For each row, count number of unique values across methods
n_unique_per_row = rounded.nunique(axis=1)

n_unique_per_row.max()

1

In [21]:
# Indexing: get row for specific CarNumber WITHOUT index

car_number = "O136HO197RUS"  # example CarNumber from the subject

row_no_index = df[df["CarNumber"] == car_number]

In [22]:
row_no_index

,CarNumber,Refund,Fines,Make,Model,Year,calc_for_loop,calc_iterrows,calc_apply,calc_series,calc_values
715,O136HO197RUS,2.0,7800.0,Toyota,Corolla,1999,7796100.0,7796100.0,7796100.0,7796100.0,7796100.0


In [23]:
# Set index to CarNumber for faster lookups
df = df.set_index("CarNumber")
df.head()

,Refund,Fines,Make,Model,Year,calc_for_loop,calc_iterrows,calc_apply,calc_series,calc_values
CarNumber,,,,,,,,,,
Y163O8161RUS,2.0,3200.0,Ford,Focus,1989,3182400.0,3182400.0,3182400.0,3182400.0,3182400.0
E432XX77RUS,1.0,6500.0,Toyota,Camry,1995,12967500.0,12967500.0,12967500.0,12967500.0,12967500.0
7184TT36RUS,1.0,2100.0,Ford,Focus,1984,4166400.0,4166400.0,4166400.0,4166400.0,4166400.0
X582HE161RUS,2.0,2000.0,Ford,Focus,2015,2015000.0,2015000.0,2015000.0,2015000.0,2015000.0
92918M178RUS,1.0,5700.0,Ford,Focus,2014,11479800.0,11479800.0,11479800.0,11479800.0,11479800.0


In [26]:
# Indexing: get row for specific CarNumber WITH index

car_number = "O136HO197RUS"

row_with_index = df.loc[car_number]

In [27]:
row_with_index

Refund                 2.0
Fines               7800.0
Make                Toyota
Model              Corolla
Year                  1999
calc_for_loop    7796100.0
calc_iterrows    7796100.0
calc_apply       7796100.0
calc_series      7796100.0
calc_values      7796100.0
Name: O136HO197RUS, dtype: object

In [28]:
# Baseline: memory usage and dtypes before optimization
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to NEWCAR05RUS
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Refund         930 non-null    float64
 1   Fines          930 non-null    float64
 2   Make           930 non-null    object 
 3   Model          919 non-null    object 
 4   Year           930 non-null    int64  
 5   calc_for_loop  930 non-null    float64
 6   calc_iterrows  930 non-null    float64
 7   calc_apply     930 non-null    float64
 8   calc_series    930 non-null    float64
 9   calc_values    930 non-null    float64
dtypes: float64(7), int64(1), object(2)
memory usage: 265.1 KB


In [29]:
# Create a copy for optimization steps
optimized_df = df.copy()

In [30]:
# Downcast float64 columns to float32

float_cols = optimized_df.select_dtypes(include=["float64"]).columns

for col in float_cols:
    optimized_df[col] = optimized_df[col].astype("float32")

In [31]:
# Downcast int64 columns to smallest possible integer dtype

int_cols = optimized_df.select_dtypes(include=["int64"]).columns

for col in int_cols:
    optimized_df[col] = pd.to_numeric(optimized_df[col], downcast="integer")

In [32]:
# Check memory usage AFTER numeric downcasting
optimized_df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to NEWCAR05RUS
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Refund         930 non-null    float32
 1   Fines          930 non-null    float32
 2   Make           930 non-null    object 
 3   Model          919 non-null    object 
 4   Year           930 non-null    int16  
 5   calc_for_loop  930 non-null    float32
 6   calc_iterrows  930 non-null    float32
 7   calc_apply     930 non-null    float32
 8   calc_series    930 non-null    float32
 9   calc_values    930 non-null    float32
dtypes: float32(7), int16(1), object(2)
memory usage: 234.2 KB


In [33]:
# Convert object columns to category to save memory

object_cols = optimized_df.select_dtypes(include=["object"]).columns

for col in object_cols:
    optimized_df[col] = optimized_df[col].astype("category")

In [34]:
# Check memory usage after converting object columns to category
optimized_df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to NEWCAR05RUS
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Refund         930 non-null    float32 
 1   Fines          930 non-null    float32 
 2   Make           930 non-null    category
 3   Model          919 non-null    category
 4   Year           930 non-null    int16   
 5   calc_for_loop  930 non-null    float32 
 6   calc_iterrows  930 non-null    float32 
 7   calc_apply     930 non-null    float32 
 8   calc_series    930 non-null    float32 
 9   calc_values    930 non-null    float32 
dtypes: category(2), float32(7), int16(1)
memory usage: 126.0 KB
